# Single-Session Preprocessing Recovery

Recovered notebook for regenerating the historical local preprocessing outputs for one session.

Outputs written by the processing cell:
- `trial_ts.pkl`
- `<session>_ks4_spike_times.pkl` (DataFrame with `unit_id` and raw sample `spike_times`)

The stimulus timing path is the historical one: analog channel thresholding (`visual=0`, `audio=1`) aligned into probe time with the shared square-wave sync.


In [ ]:
import sys
import traceback
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import spks

candidate_roots = {
    Path().resolve(),
    Path().resolve().parent,
    Path().resolve().parent.parent,
}
for path in candidate_roots:
    if (path / "ephys").exists() and str(path) not in sys.path:
        sys.path.insert(0, str(path))
        break

from ephys.src.utils.legacy_preprocessing import (  # noqa: E402
    get_cluster_spike_times,
    get_good_units,
    load_sync_data,
    process_port_events,
    process_trial_data,
)

In [ ]:
# Session configuration
mouse = "GRB006"
session = "20240821_121447"

# Raw session roots to try, in order. The first one containing the session and
# a `kilosort2.5/imec0` folder will be used.
data_root_candidates = [
    Path("/Volumes/grb_ephys/data"),
    Path("/Volumes/red_ssd/data"),
    Path("/Users/gabriel/data"),
]

# Where the recovered local outputs should be written.
output_root = Path("/Users/gabriel/data")
mirror_download_exports = False
download_export_dir = Path("/Users/gabriel/Downloads/Organized/Code")

# Safety switches
overwrite = False
write_outputs = False

In [ ]:
def resolve_session_path(mouse: str, session: str, roots: list[Path]) -> Path:
    checked = []
    for root in roots:
        session_path = root / mouse / session
        kilosort_path = session_path / "kilosort2.5" / "imec0"
        checked.append(str(session_path))
        if session_path.exists() and kilosort_path.exists():
            return session_path
    joined = "\n".join(checked)
    raise FileNotFoundError(
        f"Could not find a session root containing kilosort2.5/imec0 in:\n{joined}"
    )


def build_ks4_spike_dataframe(
    spike_times_samples: np.ndarray,
    spike_clusters: np.ndarray,
    good_unit_mask: np.ndarray,
) -> pd.DataFrame:
    unit_ids = np.unique(spike_clusters[good_unit_mask]).astype(int)
    return pd.DataFrame(
        {
            "unit_id": unit_ids,
            "spike_times": [
                spike_times_samples[good_unit_mask][
                    spike_clusters[good_unit_mask] == unit_id
                ]
                for unit_id in unit_ids
            ],
        }
    )


def process_session(
    mouse: str,
    session: str,
    *,
    data_roots: list[Path],
    output_root: Path,
    overwrite: bool = False,
    write_outputs: bool = True,
    mirror_download_exports: bool = False,
    download_export_dir: Path | None = None,
) -> dict[str, Any]:
    session_path = resolve_session_path(mouse, session, data_roots)
    kilosort_path = session_path / "kilosort2.5" / "imec0"

    save_dir = output_root / mouse / session / "pre_processed"
    trial_ts_file = save_dir / "trial_ts.pkl"
    ks4_spike_times_file = save_dir / f"{session}_ks4_spike_times.pkl"

    if write_outputs:
        save_dir.mkdir(parents=True, exist_ok=True)
        if not overwrite and trial_ts_file.exists() and ks4_spike_times_file.exists():
            print(f"Skipping {mouse} / {session}: outputs already exist.")
            return {
                "mouse": mouse,
                "session": session,
                "session_path": session_path,
                "save_dir": save_dir,
                "trial_ts_file": trial_ts_file,
                "ks4_spike_times_file": ks4_spike_times_file,
                "skipped": True,
            }

    print(f"Processing {mouse} / {session}")
    corrected_onsets, corrected_offsets, time_vector, srate, analog_signals = (
        load_sync_data(str(session_path))
    )
    trial_starts, port_events = process_port_events(
        corrected_onsets, corrected_offsets, srate
    )
    behavior_data, trial_ts, stim_ts_per_channel = process_trial_data(
        str(session_path),
        trial_starts,
        time_vector,
        srate,
        analog_signals,
        port_events,
        mouse,
        session,
    )

    spike_clusters = np.load(kilosort_path / "spike_clusters.npy")
    spike_times_samples = np.load(kilosort_path / "spike_times.npy")
    spike_times_seconds = spike_times_samples / srate

    clu = spks.clusters.Clusters(
        folder=str(kilosort_path),
        get_waveforms=False,
        get_metrics=True,
        load_template_features=True,
    )
    good_unit_mask, n_units = get_good_units(
        clusters_obj=clu, spike_clusters=spike_clusters
    )
    print(f"Good units: {n_units}")

    spike_times_per_unit = get_cluster_spike_times(
        spike_times=spike_times_seconds,
        spike_clusters=spike_clusters,
        good_unit_ids=good_unit_mask,
    )
    ks4_spike_df = build_ks4_spike_dataframe(
        spike_times_samples=spike_times_samples,
        spike_clusters=spike_clusters,
        good_unit_mask=good_unit_mask,
    )

    if write_outputs:
        trial_ts.to_pickle(trial_ts_file)
        ks4_spike_df.to_pickle(ks4_spike_times_file)
        print(f"Saved outputs to {save_dir}")

        if mirror_download_exports:
            if download_export_dir is None:
                raise ValueError(
                    "download_export_dir is required when mirroring exports"
                )
            download_export_dir.mkdir(parents=True, exist_ok=True)
            trial_ts.to_pickle(download_export_dir / "trial_ts.pkl")
            ks4_spike_df.to_pickle(
                download_export_dir / f"{session}_ks4_spike_times.pkl"
            )
            print(f"Mirrored convenience exports to {download_export_dir}")
    else:
        print("write_outputs is False; skipping file writes.")

    return {
        "mouse": mouse,
        "session": session,
        "session_path": session_path,
        "kilosort_path": kilosort_path,
        "save_dir": save_dir,
        "trial_ts_file": trial_ts_file,
        "ks4_spike_times_file": ks4_spike_times_file,
        "corrected_onsets": corrected_onsets,
        "corrected_offsets": corrected_offsets,
        "time_vector": time_vector,
        "sampling_rate": srate,
        "analog_signals": analog_signals,
        "trial_starts": trial_starts,
        "port_events": port_events,
        "behavior_data": behavior_data,
        "trial_ts": trial_ts,
        "stim_ts_per_channel": stim_ts_per_channel,
        "spike_clusters": spike_clusters,
        "spike_times_samples": spike_times_samples,
        "spike_times_seconds": spike_times_seconds,
        "good_unit_mask": good_unit_mask,
        "n_units": n_units,
        "spike_times_per_unit": spike_times_per_unit,
        "ks4_spike_df": ks4_spike_df,
    }

In [ ]:
session_outputs = {}
try:
    session_outputs = process_session(
        mouse=mouse,
        session=session,
        data_roots=data_root_candidates,
        output_root=output_root,
        overwrite=overwrite,
        write_outputs=write_outputs,
        mirror_download_exports=mirror_download_exports,
        download_export_dir=download_export_dir,
    )
except Exception:
    traceback.print_exc()
else:
    print("Processing complete.")

In [ ]:
trial_ts = session_outputs["trial_ts"]
ks4_spike_df = session_outputs["ks4_spike_df"]

print(f"trial_ts rows: {len(trial_ts)}")
print(f"ks4 rows: {len(ks4_spike_df)}")
print(f"save_dir: {session_outputs['save_dir']}")
trial_ts.head(3)

In [ ]:
time = session_outputs["time_vector"]
visual_signal = session_outputs["analog_signals"]["visual"]
audio_signal = session_outputs["analog_signals"]["audio"]

start_time = 63.3
end_time = 64.5

start_idx = int(np.searchsorted(time, start_time))
end_idx = int(np.searchsorted(time, end_time))

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(
    time[start_idx:end_idx], visual_signal[start_idx:end_idx], label="visual", alpha=0.8
)
ax.plot(
    time[start_idx:end_idx], audio_signal[start_idx:end_idx], label="audio", alpha=0.8
)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Signal amplitude (a.u.)")
ax.set_title(f"{mouse} {session}: analog stimulus channels")
ax.legend(loc="upper right")
fig.tight_layout()